In [22]:
from PIL import Image, ImageDraw, ImageFont


def draw_vertical_digits_corner(
    image: Image.Image,
    text: str,
    font_path: str,
    corner: str = "top-right",
    margin: int = 15,
    font_size: int = 22,
    color: tuple = (139, 30, 40, 255),   # тёмно-красный
    char_spacing: int = 4,
    stroke_width: int = 0,
) -> Image.Image:
    """
    Рисует текст (обычно цифры) вертикальным столбиком — символ под символом —
    в указанном углу изображения.
    """
    image = image.convert("RGBA")
    overlay = Image.new("RGBA", image.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    font = ImageFont.truetype(font_path, font_size)

    char_sizes = []
    max_w = 0
    total_h = 0
    for ch in text:
        bbox = draw.textbbox((0, 0), ch, font=font, stroke_width=stroke_width)
        w, h = bbox[2] - bbox[0], bbox[3] - bbox[1]
        char_sizes.append((w, h))
        max_w = max(max_w, w)
        total_h += h + char_spacing
    total_h -= char_spacing

    img_w, img_h = image.size
    if corner == "top-right":
        x0, y0 = img_w - margin - max_w, margin
    elif corner == "top-left":
        x0, y0 = margin, margin
    elif corner == "bottom-right":
        x0, y0 = img_w - margin - max_w, img_h - margin - total_h
    elif corner == "bottom-left":
        x0, y0 = margin, img_h - margin - total_h
    else:
        raise ValueError("corner: top-right / top-left / bottom-right / bottom-left")

    y = y0
    for ch, (w, h) in zip(text, char_sizes):
        x = x0 + (max_w - w) / 2
        draw.text((x, y), ch, font=font, fill=color, stroke_width=stroke_width)
        y += h + char_spacing

    return Image.alpha_composite(image, overlay)

In [21]:
img = Image.open("/Users/roman/projects/docs_generator/data/passport/Pasport_RF.jpg")
result = draw_vertical_digits_corner(
    img,
    text="000000",
    font_path="/Users/roman/projects/docs_generator/fonts/Arial_Italic.ttf",
    corner="top-right",
    margin=20,
    font_size=26,
    color=(139, 30, 40, 255),  # тёмно-красный, как на образце
    char_spacing=6,
)
result.convert("RGB").save("result.png")

In [ ]:
from PIL import Image, ImageDraw, ImageFont

def draw_vertical_text(
    image: Image.Image,
    text: str,
    position: tuple[int, int],
    font_path: str,
    font_size: int = 20,
    color: tuple[int, int, int, int] = (0, 0, 0, 255),
    spacing: int = 2,
    angle: float = 90,
) -> Image.Image:
    """
    Рисует текст вертикально (по одному символу друг под другом,
    либо целиком повёрнутым на angle градусов) поверх image.

    position — левый верхний угол области, где начинается текст.
    """
    font = ImageFont.truetype(font_path, font_size)

    # Вариант А: текст целиком повёрнут (как надпись сбоку)
    # 1. Рисуем текст на отдельном прозрачном слое горизонтально
    bbox = font.getbbox(text)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    txt_layer = Image.new("RGBA", (text_w + 10, text_h + 10), (0, 0, 0, 0))
    txt_draw = ImageDraw.Draw(txt_layer)
    txt_draw.text((0, 0), text, font=font, fill=color)

    # 2. Поворачиваем слой
    rotated = txt_layer.rotate(angle, expand=True)

    # 3. Накладываем на исходное изображение
    image = image.convert("RGBA")
    image.paste(rotated, position, rotated)
    return image


def draw_vertical_text_stacked(
    image: Image.Image,
    text: str,
    position: tuple[int, int],
    font_path: str,
    font_size: int = 20,
    color: tuple[int, int, int, int] = (0, 0, 0, 255),
    spacing: int = 4,
) -> Image.Image:
    """
    Вариант Б: символы идут друг под другом (столбиком), без поворота.
    """
    draw = ImageDraw.Draw(image)
    font = ImageFont.truetype(font_path, font_size)
    x, y = position
    for ch in text:
        draw.text((x, y), ch, font=font, fill=color)
        bbox = font.getbbox(ch)
        char_h = bbox[3] - bbox[1]
        y += char_h + spacing
    return image

In [35]:
img = Image.open("/Users/roman/projects/docs_generator/data/passport/Pasport_RF.jpg")
img = draw_vertical_text(
    img,
    text="123456",
    position=(50, 100),
    font_path="/Users/roman/projects/docs_generator/fonts/Arial_Italic.ttf",
    font_size=24,
    color=(50, 50, 50, 255),
    angle=90,
)
img.save("result.png")